# DAY 12 – FINAL REALISTIC PROJECT
## Student Placement Prediction System (Professional Version)

### Objective
This notebook implements a COMPLETE REAL-WORLD STYLE placement prediction system including:
- Realistic dataset
- Positive and Negative features
- Scaling vs Without Scaling comparison
- Model comparison
- Hyperparameter tuning
- Explainability
- Dynamic prediction system

This is designed exactly like an industry ML project.


## Step 1 – Import Libraries


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Step 2 – Create Realistic Dataset

Features include:
- Positive factors: CGPA, Internships, Projects, Coding Skill
- Negative factors: Backlogs, Attendance Issues, Discipline Issues


In [2]:
data = pd.DataFrame({
    'CGPA': [8.5, 6.2, 7.8, 5.5, 9.1, 6.8, 7.2, 8.0, 6.0, 5.8, 8.9, 7.4, 6.5, 9.0, 7.1, 6.3, 8.2, 5.9, 7.6, 8.7, 6.1, 7.9, 8.4, 6.7, 5.6, 7.3, 8.1, 6.4, 7.7, 8.3],

    'Internships': [2, 0, 1, 0, 3, 1, 1, 2, 0, 0, 3, 2, 1, 3, 1, 0, 2, 0, 2, 3, 0, 2, 3, 1, 0, 1, 2, 1, 2, 3],

    'Projects': [3, 1, 2, 1, 4, 2, 2, 3, 1, 1, 4, 3, 2, 4, 2, 1, 3, 1, 2, 4, 1, 3, 4, 2, 1, 2, 3, 2, 3, 4],

    'CodingSkill': [4, 2, 3, 1, 5, 3, 3, 4, 2, 1, 5, 3, 2, 5, 3, 2, 4, 1, 3, 5, 2, 4, 4, 3, 1, 3, 4, 2, 3, 4],

    'Backlogs': [0, 2, 1, 3, 0, 1, 1, 0, 2, 3, 0, 1, 2, 0, 1, 2, 0, 3, 1, 0, 2, 0, 0, 1, 3, 1, 0, 2, 1, 0],

    'AttendanceIssues': [0, 1, 0, 2, 0, 1, 0, 0, 2, 2, 0, 0, 1, 0, 1, 2, 0, 2, 1, 0, 2, 0, 0, 1, 2, 1, 0, 2, 0, 0],

    'DisciplineIssues': [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
})

data.head()

,CGPA,Internships,Projects,CodingSkill,Backlogs,AttendanceIssues,DisciplineIssues
0,8.5,2,3,4,0,0,0
1,6.2,0,1,2,2,1,0
2,7.8,1,2,3,1,0,0
3,5.5,0,1,1,3,2,1
4,9.1,3,4,5,0,0,0


## Step 3 – Rule Based Target Creation

Placement logic:
- High CGPA + Good Coding + Internships = Positive
- Backlogs, Discipline, Attendance = Negative


In [3]:
data['Placed'] = np.where(
    (data['CGPA'] >= 7.0) &
    (data['CodingSkill'] >= 3) &
    (data['Backlogs'] == 0) &
    (data['DisciplineIssues'] == 0),
    1, 0
)

data.head()

,CGPA,Internships,Projects,CodingSkill,Backlogs,AttendanceIssues,DisciplineIssues,Placed
0,8.5,2,3,4,0,0,0,1
1,6.2,0,1,2,2,1,0,0
2,7.8,1,2,3,1,0,0,0
3,5.5,0,1,1,3,2,1,0
4,9.1,3,4,5,0,0,0,1


## Step 4 – Feature Engineering


In [4]:
data['PositiveScore'] = data['CGPA'] + data['Internships'] + data['Projects'] + data['CodingSkill']
data['NegativeScore'] = data['Backlogs'] + data['AttendanceIssues'] + data['DisciplineIssues']

data.head()

,CGPA,Internships,Projects,CodingSkill,Backlogs,AttendanceIssues,DisciplineIssues,Placed,PositiveScore,NegativeScore
0,8.5,2,3,4,0,0,0,1,17.5,0
1,6.2,0,1,2,2,1,0,0,9.2,3
2,7.8,1,2,3,1,0,0,0,13.8,1
3,5.5,0,1,1,3,2,1,0,7.5,6
4,9.1,3,4,5,0,0,0,1,21.1,0


## Step 5 – Prepare X and y


In [5]:
X = data.drop('Placed', axis=1)
y = data['Placed']

## Step 6 – Train Test Split


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Step 7 – Without Scaling Model


In [7]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("WITHOUT SCALING RESULTS")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

WITHOUT SCALING RESULTS
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



## Step 8 – With Scaling Model


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LogisticRegression()
model_scaled.fit(X_train_scaled, y_train)

pred_scaled = model_scaled.predict(X_test_scaled)

print("WITH SCALING RESULTS")
print("Accuracy:", accuracy_score(y_test, pred_scaled))
print(classification_report(y_test, pred_scaled))

WITH SCALING RESULTS
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



## Step 9 – Model Comparison


In [9]:
models = {
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier()
}

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    p = m.predict(X_test_scaled)
    print(name)
    print("Accuracy:", accuracy_score(y_test, p))
    print(classification_report(y_test, p))
    print("--------------------")

Logistic Regression
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

--------------------
KNN
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

--------------------
Decision Tree
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

--------------------


## Step 10 – Hyperparameter Tuning


In [10]:
param_grid = {'C': [0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}

grid = GridSearchCV(LogisticRegression(), param_grid, cv=3)
grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_
print("Best Parameters:", grid.best_params_)

Best Parameters: {'C': 0.1, 'solver': 'liblinear'}


## Step 11 – Explainability


In [11]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Weight': best_model.coef_[0]
})

importance.sort_values(by='Weight', ascending=False)

,Feature,Weight
3,CodingSkill,0.266584
2,Projects,0.259026
7,PositiveScore,0.254001
0,CGPA,0.236373
1,Internships,0.230249
6,DisciplineIssues,-0.012290
5,AttendanceIssues,-0.172550
8,NegativeScore,-0.209615
4,Backlogs,-0.280714


## Step 12 – FINAL DYNAMIC PREDICTION SYSTEM

Change values and prediction will change automatically!


In [12]:
def predict_student(cgpa, internships, projects, coding, backlogs, attendance, discipline):
    positive = cgpa + internships + projects + coding
    negative = backlogs + attendance + discipline

    input_data = np.array([[cgpa, internships, projects, coding, backlogs, attendance, discipline, positive, negative]])
    input_scaled = scaler.transform(input_data)

    result = best_model.predict(input_scaled)

    if result[0] == 1:
        print("FINAL PREDICTION: PLACED")
    else:
        print("FINAL PREDICTION: NOT PLACED")

## Try Different Values Below


In [13]:
predict_student(
    cgpa=7.2,
    internships=2,
    projects=3,
    coding=4,
    backlogs=1,
    attendance=0,
    discipline=0
)

FINAL PREDICTION: PLACED


C:\Users\VICTUS\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
